# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [2]:
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva.
# Mostrar los tipos y los valores que no pudieron convertirse.
# Desarrollo de la actividad 1
eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip().str.zfill(2)
eva["anio"] = pd.to_numeric(eva["Año"], errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"], errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"], errors="coerce")

# Mostrar tipos resultantes
print("Tipos de las columnas preparadas:")
print(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].dtypes)

# Registrar ausencias resultantes (NaN)
print("\nConteo de valores que no pudieron convertirse (nulos):")
print(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].isna().sum())

# Conservación de filas original
print("\nTotal de registros de la tabla EVA:", len(eva))

Tipos de las columnas preparadas:
codigo_departamento     object
anio                     Int64
area_ha                float64
produccion_t           float64
dtype: object

Conteo de valores que no pudieron convertirse (nulos):
codigo_departamento    0
anio                   0
area_ha                0
produccion_t           0
dtype: int64

Total de registros de la tabla EVA: 166732


**Interpretación del resultado:**

_Las columnas fueron preparadas exitosamente conservando los 166.732 registros de entrada. El código departamental se limpió de espacios y rellenó con ceros usando .zfill(2) para mantenerlo como texto. Las variables de año, área y producción pasaron a formatos numéricos (Int64 y float64) convirtiendo los errores (como la cadena "ND") en valores nulos (NaN o <NA>), tal como lo recomienda el manual para evitar alterar los promedios estadísticos._

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [3]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.
# Mostrar los conteos de entrada, selección y diferencia.
# Desarrollo de la actividad 2
mascara = (
    eva["codigo_departamento"].eq("15") & 
    eva["anio"].isin([2023, 2024]) & 
    eva["area_ha"].gt(0) & 
    eva["produccion_t"].ge(0)
)

consulta_loc = eva.loc[mascara].copy()

total_original = len(eva)
total_seleccionado = len(consulta_loc)
diferencia = total_original - total_seleccionado

print(f"Total de entrada original: {total_original}")
print(f"Total de registros seleccionados: {total_seleccionado}")
print(f"Diferencia (filas excluidas): {diferencia}")

Total de entrada original: 166732
Total de registros seleccionados: 4924
Diferencia (filas excluidas): 161808


**Interpretación del resultado:**

_Se construyó una máscara booleana encadenada con los operadores lógicos &. Al utilizar el método .copy(), garantizamos que consulta_loc sea independiente de eva y evitamos el error SettingWithCopyWarning en el futuro. La diferencia representa registros legítimamente ignorados por estar en otros departamentos (diferentes a Boyacá), en años distintos, o por no tener superficie cosechada ni producción positiva._

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [4]:
# Desarrollo de la actividad 3
# Construir consulta_query.
# Comprobar que consulta_loc y consulta_query son equivalentes.
# Desarrollo de la actividad 3
cod_boyaca = "15"
anios_obj = [2023, 2024]

consulta_query = eva.query(
    "codigo_departamento == @cod_boyaca and "
    "anio in @anios_obj and "
    "area_ha > 0 and "
    "produccion_t >= 0",
    engine="python"
).copy()

# Comparación directa
pd.testing.assert_frame_equal(consulta_loc, consulta_query)
print("¡Éxito! consulta_loc y consulta_query son idénticas en filas, columnas y valores.")

¡Éxito! consulta_loc y consulta_query son idénticas en filas, columnas y valores.


**Interpretación del resultado:**

_El método .query() demostró ser totalmente equivalente a la máscara de .loc, pero con una sintaxis más limpia en forma de cadena de texto y utilizando variables de Python a través del prefijo @. La prueba silenciosa de assert_frame_equal confirma que ambas vías extrajeron la misma muestra de datos exactos sin discrepancias._

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [5]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
# Verificar los totales y los porcentajes por año.
# Desarrollo de la actividad 4
resumen = (
    consulta_loc.groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

# Calcular total por año
resumen["total_anio"] = (
    resumen.groupby("anio", dropna=False)["registros"].transform("sum")
)

# Calcular porcentaje
resumen["porcentaje_anual"] = (
    resumen["registros"] / resumen["total_anio"] * 100
)

# Ordenar los datos
resumen = resumen.sort_values(["anio", "registros"], ascending=[True, False])

display(resumen.head(10))

# Controles de calidad
print("Conciliación de registros totales:", resumen["registros"].sum() == len(consulta_loc))
print("\nSuma porcentual por año:")
print(resumen.groupby("anio", dropna=False)["porcentaje_anual"].sum())

,anio,Cultivo,registros,total_anio,porcentaje_anual
70,2023,Papa,239,2436,9.811166
62,2023,Maíz,194,2436,7.963875
9,2023,Arveja,176,2436,7.224959
38,2023,Frijol,162,2436,6.650246
88,2023,Tomate,113,2436,4.638752
22,2023,Cebolla de bulbo,93,2436,3.817734
93,2023,Yuca,82,2436,3.366174
20,2023,Caña,63,2436,2.586207
90,2023,Trigo,58,2436,2.380952
64,2023,Mora,51,2436,2.093596


Conciliación de registros totales: True

Suma porcentual por año:
anio
2023    100.0
2024    100.0
Name: porcentaje_anual, dtype: float64


**Interpretación del resultado:**

_Se empleó la función transform("sum") sobre la agrupación por año, lo que permite replicar el subtotal anual en cada fila del grupo sin distorsionar la estructura. La validación afirma que no hemos perdido registros frente a consulta_loc y las sumas porcentuales llegan perfectamente al 100% en cada año, distribuyendo correctamente las filas dentro de Boyacá._

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [6]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
# Exportar resumen y explicar dos resultados en la celda siguiente.
# Desarrollo de la actividad 5
tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    fill_value=0,
    aggfunc="sum"
)

display(tabla_dinamica.head(10))

# Exportar a CSV
resumen.to_csv(OUT / "resumen_eva.csv", index=False)
print("Archivo exportado correctamente a:", OUT / "resumen_eva.csv")

anio,2023,2024
Cultivo,,
Acelga,5,8
Achira,4,4
Agraz - mortiño,3,4
Aguacate,48,52
Ahuyama,23,26
Ajo,7,7
Ají,1,2
Apio,0,1
Arazá,3,4


Archivo exportado correctamente a: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv


**Interpretación del resultado:**

_*Conclusión 1 (Composición agrícola): La matriz dinámica muestra rápidamente la distribución de cultivos registrados en Boyacá. Los ceros generados por fill_value=0 indican que para un cultivo específico, no hubo ningún registro de evaluación agrícola (EVA) capturado en el territorio para ese año en particular.
*Conclusión 2 (Limitación interpretativa): Es imperativo aclarar que la columna porcentaje_anual calculada representa la "proporción de registros ingresados a la base de datos", y no equivale al porcentaje de hectáreas (área física) cultivadas en Boyacá. Un solo registro con miles de hectáreas pesa lo mismo en el conteo que un registro de media hectárea._

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.